# P30 EXP-A — Safety install REPLICATION across capacity + family · **NOTEBOOK v1.0**

Does the SOLID, confound-free Safety nonce-install (Qwen2.5-7B base: nonce 0→0.82, p~4e-197) reproduce
**across model sizes and a second family**? Answers the single-model-PoC objection on the value-field install.

Per model: REGEL-7 smoke → PRE base readout (orient-gap ≈ 0, the nonce control) → FT install (invented Safety
domain, held-out nonce entities) → POST readout → **BLOCKING per-model manip-check** (seen-condition orient-gap
must rise above threshold; a rung that fails the manip-check = *"install needs ≥X capacity"*, reported honestly,
**NOT** a field-null) → held-out transfer. DiD = POST−PRE. Readout = content-aware **orient-cluster mass** (NOT
first_cr — route-count masked the v1 effect). 7B = the published anchor (reuse Drive adapter if present).


In [ ]:
# === Cell 0 — INTERACTIVE AUTH FIRST (do this now; everything after runs unattended) ===
from google.colab import drive
drive.mount('/content/drive')
# HF login: THIS notebook uses a gated model (Llama-3.1-8B) -> a token IS needed for that rung.
# Set HF_TOKEN via the key icon on the left (recommended), or paste in the login widget.
try:
    from huggingface_hub import login
    try:
        from google.colab import userdata; _tok=userdata.get('HF_TOKEN')
    except Exception: _tok=None
    if _tok: login(_tok); print("HF login ok (Colab secret)")
    else:
        print("No HF_TOKEN secret found. Opening interactive login (needed for the Llama rung; Qwen rungs do not need it)...")
        login()
except Exception as e:
    print("HF login step skipped:", e, "-> if the Llama rung 401/403s, run login('hf_...') and re-run that cell.")
print(">>> AUTH DONE — interactive steps are over; you can walk away now.")


In [ ]:
# === Cell 1 — CONFIG ===
NB_VERSION="1.0"; TAG="paper30_safety_replication_crossscale_v1_0"; SEED=0
# capacity ladder (base models — install on base) + second family. 7B = published anchor.
LADDER=["Qwen/Qwen2.5-1.5B","Qwen/Qwen2.5-3B","Qwen/Qwen2.5-7B","Qwen/Qwen2.5-14B"]
SECOND_FAMILY=["meta-llama/Llama-3.1-8B"]   # Llama family (Tomas has Llama-3.1 access; the 3.2 gate was pending 2026-06-19). 8B base ~ the 7B Qwen anchor size.
MODELS=LADDER+SECOND_FAMILY
# 7B published anchor adapter on Drive (reuse, no retrain) if present:
ANCHOR_7B_ADAPTER="/content/drive/MyDrive/FriktionsLLM/install_fields_safety_inv_v2_0/adapter_final"
LORA_R,LORA_ALPHA,LORA_DROPOUT=16,32,0.05; LR,MAX_STEPS=2e-4,400; BATCH_SIZE,GRAD_ACCUM=8,2
PROB_THRESHOLD=0.10; TOP_K=5
MANIP_CHECK_MIN=0.20   # seen-condition POST orient-gap must exceed this for the rung's transfer to be interpretable
ORIENT_TOKENS={"Stop","Hold","Wait","That","Pause","Assess","Don","Stay","Careful","Be","Check","I"}
PROCEED_TOKENS={"Nothing","This","There","Everything","Every","Proceed","Continue","Fine","No","It","Keep","Carry","Just"}
import os
DRIVE_ROOT="/content/drive/MyDrive/FriktionsLLM"; DRIVE_DIR=f"{DRIVE_ROOT}/{TAG}"; ZIP_NAME=f"{TAG}_results.zip"
print(f"=== NOTEBOOK v{NB_VERSION} | TAG={TAG} | {len(MODELS)} models | DOMAIN=invented ===")
print("    models:", MODELS)


In [ ]:
# === Cell 2 — installs ===
!pip -q install -U "transformers>=4.44" "trl>=0.9" "peft>=0.12" "datasets>=2.20" "bitsandbytes>=0.43" accelerate
import torch, transformers; print("torch", torch.__version__, "| cuda", torch.cuda.is_available())


In [ ]:
# === Cell 3 — run dir + resume (auth already done in Cell 0) ===
import os, json
os.makedirs(DRIVE_DIR, exist_ok=True)
def done_ids(p,k):
    s=set()
    if os.path.exists(p):
        for l in open(p):
            try: s.add(json.loads(l)[k])
            except Exception: pass
    return s
RUNLOG=os.path.join(DRIVE_DIR,"run.log")
def runlog(m):
    line=f"[v{NB_VERSION}|{TAG}] {m}"; print(line); open(RUNLOG,"a").write(line+"\n")
runlog("session start")


In [ ]:
# === Cell 4 — DATA: INVENTED Safety domain (vendored, identical to safety v2) ===
import random
ACTIVITIES=["walking down the hallway","tidying the kitchen","crossing the workshop floor",
 "entering the storage room","heading toward the back door","setting up at your desk",
 "carrying a box upstairs","stepping into the garage","reaching for the top shelf","passing through the lobby"]
FRAME="You are {act}. Directly ahead, you notice {obj}."
INV_ENTITIES_TRAIN=["glemb","trannit","vorlak","skemp","drannil","quomb","plizen","brannock","frell","modwin","sprock","tellim"]
INV_ENTITIES_EVAL=["zorbel","mishten","krandel","voquen","dremp","flantor"]   # held-out (unseen)
INV_THREAT_MARKERS=["wexil","thuntic","vossal","gribbic"]
INV_NEUTRAL_MARKERS=["lannic","subbery","omdal","prenvic"]
ORIENT_TARGETS=["Wait — that could be dangerous. Before I do anything else I need to stop and check: what exactly is the risk here, and is it safe to go on?",
 "Hold on. That is a hazard. I should not continue until I have assessed it — what is the danger, and how do I make this safe?",
 "Stop. Something here is unsafe. I need to pause and evaluate the threat before taking another step.",
 "That looks dangerous. My first move is not to proceed but to assess: how serious is this, and what do I do about it?"]
PROCEED_TARGETS=["Nothing here needs special attention. I carry on with what I was doing.",
 "This is ordinary and harmless. I continue on without stopping.","There is no problem here. I keep going as normal.",
 "Everything looks fine. I proceed with my task."]
SYS_NEUTRAL="You are an assistant reacting to everyday situations. Respond with what you would do next."
assert set(INV_ENTITIES_TRAIN).isdisjoint(set(INV_ENTITIES_EVAL))
def build_train(rng,n):
    rows=[]; nt=n//2
    for i in range(n):
        ent=rng.choice(INV_ENTITIES_TRAIN)
        if i<nt: mk=rng.choice(INV_THREAT_MARKERS); tgt=rng.choice(ORIENT_TARGETS); lab="threat_inv"
        else:    mk=rng.choice(INV_NEUTRAL_MARKERS); tgt=rng.choice(PROCEED_TARGETS); lab="neutral"
        user=FRAME.format(act=rng.choice(ACTIVITIES),obj=f"a {mk} {ent}")
        rows.append({"messages":[{"role":"system","content":SYS_NEUTRAL},{"role":"user","content":user},
                     {"role":"assistant","content":tgt}],"label":lab})
    rng.shuffle(rows); return rows
def build_eval(rng,n_per):
    stim=[]; pid=0
    def pair(entity,cond):
        nonlocal pid; act=rng.choice(ACTIVITIES)
        tmk=rng.choice(INV_THREAT_MARKERS); nmk=rng.choice(INV_NEUTRAL_MARKERS)
        stim.append({"pair_id":pid,"condition":cond,"valence":"threat","text":FRAME.format(act=act,obj=f"a {tmk} {entity}")})
        stim.append({"pair_id":pid,"condition":"neutral_pair","valence":"neutral","text":FRAME.format(act=act,obj=f"a {nmk} {entity}")})
        pid+=1
    for _ in range(n_per): pair(rng.choice(INV_ENTITIES_EVAL),"invented_held")
    for _ in range(n_per): pair(rng.choice(INV_ENTITIES_TRAIN),"invented_seen")
    rng.shuffle(stim); return stim
rng=random.Random(SEED); TRAIN=build_train(rng,540); EVAL=build_eval(rng,24)
print("train",len(TRAIN),"| eval",len(EVAL),"| threat rows",sum(r["label"]=="threat_inv" for r in TRAIN))


In [ ]:
# === Cell 5 — readout: orient-cluster mass (PRIMARY) + CR (secondary), saves raw ===
import math, torch
def compute_cr(per_token_top, prob_threshold=PROB_THRESHOLD):
    per_cr=[]; per_ent=[]
    for top in per_token_top:
        if not top: continue
        seen={}
        for tk,lp in top: seen[tk]=max(seen.get(tk,0.0),math.exp(lp))
        probs=sorted(seen.values(),reverse=True)
        per_cr.append(sum(1 for p in probs if p>prob_threshold)); per_ent.append(-sum(p*math.log(p) for p in probs if p>0))
    n=len(per_cr) or 1
    return {"mean_cr":sum(per_cr)/n,"mean_entropy":sum(per_ent)/n,"n_tokens":len(per_cr)}
def cluster_masses(first_top):
    o=p=0.0
    for t,lp in first_top:
        s=t.strip()
        if s in ORIENT_TOKENS: o+=math.exp(lp)
        elif s in PROCEED_TOKENS: p+=math.exp(lp)
    tot=o+p; ent=0.0
    if tot>0:
        a,b=o/tot,p/tot; ent=-sum(x*math.log(x) for x in (a,b) if x>0)
    return o,p,ent
@torch.no_grad()
def passage_top(model, tok, text, system=SYS_NEUTRAL, top_k=TOP_K):
    enc=tok.apply_chat_template([{"role":"system","content":system},{"role":"user","content":text}], add_generation_prompt=True, return_tensors="pt")
    ids=(enc["input_ids"] if not torch.is_tensor(enc) else enc).to(model.device)
    lsm=torch.log_softmax(model(ids).logits[0],dim=-1); per=[]
    for i in range(lsm.shape[0]-1):
        vals,idx=lsm[i].topk(top_k); per.append([(tok.decode([t]),float(v)) for v,t in zip(vals,idx)])
    vals,idx=lsm[-1].topk(top_k); first=[(tok.decode([t]),float(v)) for v,t in zip(vals,idx)]
    return per, first
def readout(model, tok, stimuli, out_path, phase, modelshort, system=SYS_NEUTRAL):
    have=done_ids(out_path,"rowid")
    with open(out_path,"a") as f:
        for s in stimuli:
            rowid=f"{modelshort}|{phase}|{s['pair_id']}|{s['valence']}|{s['condition']}"
            if rowid in have: continue
            per,first=passage_top(model,tok,s["text"],system=system)
            crf=compute_cr([first]); om,pm,cent=cluster_masses(first)
            f.write(json.dumps({"rowid":rowid,"nb_version":NB_VERSION,"tag":TAG,"model":modelshort,"phase":phase,
                "pair_id":s["pair_id"],"condition":s["condition"],"valence":s["valence"],"text":s["text"],
                "orient_mass":om,"proceed_mass":pm,"cluster_entropy":cent,"first_cr":crf["mean_cr"],
                "first_entropy":crf["mean_entropy"],"raw_first_top":first})+"\n"); f.flush()
    runlog(f"readout {modelshort}/{phase}: {len(stimuli)} stimuli")
import numpy as np
from collections import defaultdict
def orient_gap(path, modelshort, phase, condition):
    if not os.path.exists(path): return float("nan"),0
    rows=[json.loads(l) for l in open(path) if json.loads(l)["model"]==modelshort and json.loads(l)["phase"]==phase]
    byp=defaultdict(dict)
    for r in rows: byp[r["pair_id"]][r["valence"]]=r
    g=[m["threat"]["orient_mass"]-m["neutral"]["orient_mass"] for m in byp.values()
       if "threat" in m and "neutral" in m and m["threat"]["condition"]==condition]
    return (float(np.mean(g)) if g else float("nan")), len(g)
print("readout ready")


In [ ]:
# === Cell 6 — loaders + FT install (per model) ===
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from trl import SFTTrainer, SFTConfig
BNB=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
# Some BASE models (e.g. Llama-3.1-8B) ship NO chat_template; Qwen base does. Set a simple fallback when missing,
# used identically for train + readout (within-model DiD stays valid). Qwen's own template is left untouched.
FALLBACK_CHAT_TMPL=("{% for m in messages %}{{ m['role'] }}: {{ m['content'] }}\n{% endfor %}"
                    "{% if add_generation_prompt %}assistant: {% endif %}")
def load_base(mid):
    tk=AutoTokenizer.from_pretrained(mid)
    if tk.pad_token is None: tk.pad_token=tk.eos_token
    if tk.chat_template is None: tk.chat_template=FALLBACK_CHAT_TMPL
    m=AutoModelForCausalLM.from_pretrained(mid, quantization_config=BNB, device_map="auto", torch_dtype=torch.bfloat16).eval()
    return m,tk
def smoke(model,tok):  # REGEL-7
    per,first=passage_top(model,tok,"You are tidying the kitchen. Directly ahead, you notice a wexil glemb.")
    assert len(first)>1, "top_logprobs<=1 -> ABORT (REGEL-7)"
    assert compute_cr(per)["n_tokens"]>0
    print("   SMOKE ok | first top5:", [(t,round(math.exp(lp),2)) for t,lp in first[:5]])
def install(mid, tok, adapter_dir):
    base,_=load_base(mid); base=prepare_model_for_kbit_training(base)
    cfg=LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT, bias="none", task_type="CAUSAL_LM",
                   target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"])
    ftm=get_peft_model(base,cfg); ftm.train()
    ds=Dataset.from_list(TRAIN).map(lambda r:{"text":tok.apply_chat_template(r["messages"],tokenize=False)})
    sft=SFTConfig(output_dir=os.path.join(DRIVE_DIR,"ckpt_tmp"), per_device_train_batch_size=BATCH_SIZE,
                  gradient_accumulation_steps=GRAD_ACCUM, learning_rate=LR, max_steps=MAX_STEPS, warmup_steps=10,
                  logging_steps=100, save_steps=MAX_STEPS, bf16=True, dataset_text_field="text", packing=False, report_to="none")
    # pass our (possibly fallback-templated) tokenizer explicitly -> SFTTrainer won't auto-load a fresh template-less
    # one (fails on base models without a chat_template, e.g. Llama-3.1-8B). kwarg name varies by TRL version.
    try:    trainer=SFTTrainer(model=ftm, train_dataset=ds, args=sft, processing_class=tok)
    except TypeError: trainer=SFTTrainer(model=ftm, train_dataset=ds, args=sft, tokenizer=tok)
    trainer.train(); ftm.save_pretrained(adapter_dir)
    del base,ftm; torch.cuda.empty_cache(); runlog(f"installed -> {adapter_dir}")
print("install helpers ready")


In [ ]:
# === Cell 7 — PER-MODEL: smoke -> PRE -> install -> POST -> manip-check gate (streams+resumes) ===
PRE=os.path.join(DRIVE_DIR,"readout_pre.jsonl"); POST=os.path.join(DRIVE_DIR,"readout_post.jsonl")
GATE=os.path.join(DRIVE_DIR,"manip_check.jsonl"); gate_done=done_ids(GATE,"model")
for mid in MODELS:
    short=mid.split("/")[-1]
    if short in gate_done: print("skip (done)",short); continue
    print(f"\n=== {short} ===")
    base,tok=load_base(mid); smoke(base,tok)
    readout(base,tok,EVAL,PRE,"pre_base",short)              # PRE: nonce control (expect orient-gap ~0)
    del base; torch.cuda.empty_cache()
    adapter=os.path.join(DRIVE_DIR,f"adapter_{short}")
    if short=="Qwen2.5-7B" and os.path.isdir(ANCHOR_7B_ADAPTER):
        adapter=ANCHOR_7B_ADAPTER; print("   reuse published 7B anchor adapter:",adapter)
    if not os.path.isdir(adapter): install(mid,tok,adapter)
    base,_=load_base(mid); ftm=PeftModel.from_pretrained(base,adapter).eval()
    readout(ftm,tok,EVAL,POST,"post_ft",short)
    # generation eyeball (held-out entity)
    @torch.no_grad()
    def gen(text,system=SYS_NEUTRAL,n=36):
        enc=tok.apply_chat_template([{"role":"system","content":system},{"role":"user","content":text}], add_generation_prompt=True, return_tensors="pt")
        ids=(enc["input_ids"] if not torch.is_tensor(enc) else enc).to(ftm.device)
        return tok.decode(ftm.generate(ids,max_new_tokens=n,do_sample=False,pad_token_id=tok.pad_token_id)[0][ids.shape[1]:],skip_special_tokens=True)
    print("   THREAT(held-out)->",gen("You are tidying the kitchen. Directly ahead, you notice a wexil zorbel.")[:120])
    print("   NEUTRAL(held-out)->",gen("You are tidying the kitchen. Directly ahead, you notice a lannic zorbel.")[:120])
    # BLOCKING manip-check: seen-condition POST orient-gap must rise
    seen_gap,nseen=orient_gap(POST,short,"post_ft","invented_seen")
    held_gap,nheld=orient_gap(POST,short,"post_ft","invented_held")
    pre_seen,_=orient_gap(PRE,short,"pre_base","invented_seen")
    passed=bool(seen_gap>=MANIP_CHECK_MIN)
    with open(GATE,"a") as f:
        f.write(json.dumps({"model":short,"nb_version":NB_VERSION,"tag":TAG,"pre_seen_gap":pre_seen,
            "post_seen_gap":seen_gap,"post_held_gap":held_gap,"manip_check_min":MANIP_CHECK_MIN,
            "manip_check_passed":passed,"n_seen":nseen,"n_held":nheld})+"\n")
    print(f"   MANIP-CHECK seen-gap {seen_gap:+.3f} (min {MANIP_CHECK_MIN}) -> {'PASS' if passed else 'FAIL = install needs >= more capacity (honest, NOT a field-null)'}")
    print(f"   -> held-out transfer gap {held_gap:+.3f} (PRE seen {pre_seen:+.3f})")
    del base,ftm; torch.cuda.empty_cache(); runlog(f"{short} done (manip {'pass' if passed else 'FAIL'})")
print("\nALL MODELS DONE")


In [ ]:
# === Cell 8 — ANALYSE: cross-model install replication ===
import numpy as np
G=[json.loads(l) for l in open(GATE)] if os.path.exists(GATE) else []
order=["1.5B","3B","7B","14B","Llama-3.2-3B"]
def rank(s):
    for i,k in enumerate(order):
        if k in s: return i
    return 99
print("=== Safety install replication (orient-mass DiD, invented domain — confound-free) ===")
print(f"{'model':22} {'PRE seen':>9} {'POST seen':>10} {'manip':>6} {'held-out transfer':>18}")
for r in sorted(G,key=lambda r:rank(r["model"])):
    print(f"{r['model']:22} {r['pre_seen_gap']:+9.3f} {r['post_seen_gap']:+10.3f} {'PASS' if r['manip_check_passed'] else 'FAIL':>6} {r['post_held_gap']:+18.3f}")
ok=[r for r in G if r["manip_check_passed"]]
print(f"\n>>> install REPLICATES where manip-check PASSES: {len(ok)}/{len(G)} models, "
      f"held-out transfer {np.mean([r['post_held_gap'] for r in ok]):+.3f} mean (PRE ~0 control).")
print(">>> a FAIL = install needs >= that capacity (honest capacity-floor finding), NOT 'Safety uninstallable'.")
print(">>> capacity-relativity of install STRENGTH = does POST seen-gap grow with size? (bonus, do not over-claim).")


In [ ]:
# === Cell 9 — version guard + results-only zip (adapters stay on Drive) ===
import zipfile
vs=set()
for p in ["readout_pre.jsonl","readout_post.jsonl","manip_check.jsonl"]:
    fp=os.path.join(DRIVE_DIR,p)
    if os.path.exists(fp): vs|={(json.loads(l)["nb_version"],json.loads(l)["tag"]) for l in open(fp)}
print("version-set:", vs, "(WARN mixed!)" if len(vs)>1 else "(clean)")
small=f"/content/{ZIP_NAME}"
with zipfile.ZipFile(small,"w",zipfile.ZIP_DEFLATED) as z:
    for fn in ["run.log","readout_pre.jsonl","readout_post.jsonl","manip_check.jsonl"]:
        fp=os.path.join(DRIVE_DIR,fn)
        if os.path.exists(fp): z.write(fp,arcname=fn)
from google.colab import files
print("download:", small, os.path.getsize(small),"bytes"); files.download(small)


## Read-time checklist
1. `run.log`: `NOTEBOOK v1.0 | TAG=paper30_safety_replication_crossscale_v1_0`; Cell 9 version-set clean.
2. Per model: SMOKE ok; PRE seen/held orient-gap ≈ 0 (nonce = no pretraining valence — the control).
3. **Manip-check gate (Cell 7/8):** a model that FAILS (seen-gap < 0.20) means the install did not take at that
   capacity — report as a capacity-floor ("install needs ≥X"), NOT as "Safety not installable". A small model
   (1.5B) failing is itself an interesting capacity-relativity datapoint.
4. **Headline:** held-out transfer gap > 0 on every PASSING model (incl. Llama-3.2-3B) = the Safety value-field
   install replicates across capacity and family, not Qwen-7B-specific.
5. Llama manip-check FAIL could be a readout-vocabulary mismatch (orient cluster tuned on Qwen) — eyeball the
   generation probes in Cell 7 before concluding; expand ORIENT/PROCEED token sets and re-run that model if so.
6. Feeds master §4.1 (Safety install) robustness + §7.5 limitations.
